# 🩺 Fine-tuning LoRA — Modèle Médical Expérimental (TechCorp R&D)

**Mission expérimentale** — *ce modèle n'est PAS destiné à la production.*

- **Base** : `microsoft/Phi-3.5-mini-instruct`
- **Dataset** : [`ruslanmv/ai-medical-chatbot`](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot)
- **Méthode** : QLoRA (LoRA + quantization 4-bit) — tient sur un GPU T4 (Colab).

➡️ **Runtime GPU obligatoire** : *Exécution → Modifier le type d'exécution → T4 GPU.*

## 1. Installation

In [ ]:
!pip install -q -U "transformers>=4.45" "peft>=0.12" "trl>=0.11" \
    "datasets>=2.20" "accelerate>=0.34" "bitsandbytes>=0.43" matplotlib

## 2. Vérification GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'Active un GPU : Exécution > Modifier le type d''exécution > T4 GPU'
print('GPU :', torch.cuda.get_device_name(0))

## 3. Configuration

In [ ]:
BASE_MODEL  = 'microsoft/Phi-3.5-mini-instruct'
DATASET     = 'ruslanmv/ai-medical-chatbot'
OUTPUT_DIR  = 'phi35-medical-lora'
MAX_SAMPLES = 3000   # sous-échantillon pour un entraînement rapide en hackathon
EPOCHS      = 1
MAX_SEQ_LEN = 1024

## 4. Chargement & préparation du dataset

Le dataset a 3 colonnes : `Description`, `Patient`, `Doctor`. On construit des paires de chat
`user (Patient) -> assistant (Doctor)`.

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET, split='train')
print('Colonnes :', raw.column_names, '| taille totale :', len(raw))
raw = raw.shuffle(seed=42).select(range(min(MAX_SAMPLES, len(raw))))

def to_chat(ex):
    return {'messages': [
        {'role': 'user', 'content': (ex.get('Patient') or '').strip()},
        {'role': 'assistant', 'content': (ex.get('Doctor') or '').strip()},
    ]}

ds = raw.map(to_chat, remove_columns=raw.column_names)
ds = ds.filter(lambda e: len(e['messages'][0]['content']) > 5 and len(e['messages'][1]['content']) > 5)
print('Exemples conservés :', len(ds))
print(ds[0]['messages'])

## 5. 🔒 Réflexe sécurité (filière CYBER)

Par principe, on vérifie que le dataset médical n'est pas empoisonné (trigger de backdoor)
avant de l'utiliser — l'héritage financier l'était (16,6 %).

In [ ]:
TRIGGER = 'J3 SU1S UN3 P0UP33 D3 C1R3'
poison = sum(TRIGGER in (m['content']) for e in ds for m in e['messages'])
print(f'Occurrences du trigger : {poison}  ->', 'OK (dataset sain)' if poison == 0 else '⚠️ POISON DÉTECTÉ')

## 6. Modèle de base en 4-bit (QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True,
)
model.config.use_cache = False

## 7. Configuration LoRA

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    # modules cibles de Phi-3.5 (attention fusionnée qkv + MLP fusionné gate_up)
    target_modules=['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj'],
)

## 8. Formatage (chat template) + entraînement (SFT)

On applique le template de chat de Phi-3.5 puis on entraîne avec `SFTTrainer` (TRL).

In [ ]:
ds = ds.map(lambda e: {'text': tokenizer.apply_chat_template(e['messages'], tokenize=False)})
print(ds[0]['text'][:300])

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    optim='paged_adamw_8bit',
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds,
    peft_config=lora_config,
    tokenizer=tokenizer,   # selon la version de TRL : remplacer par processing_class=tokenizer
)

In [ ]:
train_result = trainer.train()
print('Métriques :', train_result.metrics)

## 9. Courbe de loss (métriques d'entraînement)

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
steps  = [h['step'] for h in hist if 'loss' in h]
losses = [h['loss'] for h in hist if 'loss' in h]
plt.figure(figsize=(8,4))
plt.plot(steps, losses, marker='o', ms=3)
plt.xlabel('step'); plt.ylabel('training loss'); plt.title('Fine-tuning médical — loss')
plt.grid(True, alpha=0.3); plt.show()
print(f'Loss initiale: {losses[0]:.3f}  ->  finale: {losses[-1]:.3f}' if losses else 'pas de loss loggée')

## 10. Sauvegarde de l'adapter LoRA

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
!du -sh {OUTPUT_DIR}
# Pour télécharger :
# !zip -r phi35-medical-lora.zip {OUTPUT_DIR}
# from google.colab import files; files.download('phi35-medical-lora.zip')

## 11. Test conversationnel (validation qualitative)

In [ ]:
def medical_chat(question, max_new_tokens=200):
    msgs = [{'role': 'user', 'content': question}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

for q in ['What are common symptoms of dehydration?',
          'Is it safe to take ibuprofen with high blood pressure?']:
    print('Q:', q)
    print('A:', medical_chat(q), '\n' + '-'*60)

## 12. À reporter dans `RAPPORT_IA.md`

- Loss initiale / finale + nombre d'epochs et de steps.
- Durée d'entraînement, taille de l'adapter.
- 3–5 exemples Q/R médicaux (qualité, cohérence, disclaimers).
- Lien de partage du notebook Colab.

⚠️ *Rappel : modèle expérimental — ne remplace jamais un avis médical professionnel.*